# Transcribing Reel Audio

Test different whisper models

In [2]:
import pandas as pd
import subprocess
import shutil

In [3]:
reels = pd.read_csv("../data/videos.csv")
reels

,video_id,description,topic,speech,visual,text,source_url
0,reel_001,Volleyball jump technique,volleyball,1,1,1,https://www.instagram.com/reel/DT3K2rejzZu/?ut...
1,reel_002,Volleyball professional gameplay,volleyball,1,1,0,https://www.instagram.com/reel/DVva95sAAuA/?ut...
2,reel_003,Volleyball hitting advice,volleyball,1,1,1,https://www.instagram.com/reel/DW87lRmPs_L/?ut...
3,reel_004,Slow motion volleyball spike,volleyball,0,1,0,https://www.instagram.com/reel/DVAWEg4Efiw/?ut...
4,reel_005,Data science interview advice,data science,1,1,1,https://www.instagram.com/reel/DdeYhk2xxAH/?ut...
5,reel_006,Machine Learning project question,data science,0,0,1,https://www.instagram.com/reel/DR079PeAvcT/?ut...
6,reel_007,Answering data science interview question,data science,1,1,1,https://www.instagram.com/reel/DTlp-oXkR63/?ut...
7,reel_008,Man doing funny scream making others laugh,humour,1,0,0,https://www.instagram.com/reel/DS6W6KBgYxu/?ut...
8,reel_009,Kid makes a funny logical mistake,humour,1,1,0,https://www.instagram.com/reel/DZgKmJpOelr/?ut...
9,reel_010,Tennis one-handed backhand fixes,tennis,1,1,0,https://www.instagram.com/reel/Dd9spRwCvrW/?ut...


In [4]:
# loops over reels and extracts audio into wav files into data/audio

reel_names = reels["video_id"]
reel_names_list = reel_names.to_list()

for reel in reel_names_list:
    reel_path = f"../data/raw/{reel}.mp4"
    audio_path = f"../data/audio/{reel}.wav"

    subprocess.run(
    ["ffmpeg", "-y", "-i", reel_path,
     "-ac", "1", "-ar", "16000", "-vn", audio_path],
    check=True,
)

ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with clang version 20.1.8
  configuration: --prefix=/Users/nicolaszhu/miniforge3/envs/reel-recs --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --enable-version3 --enable-zlib --enable-libvorbis --enable-libopus --enable-libwebp --enable-libshaderc --disable-ffplay --disable-static --disable-gpl --disable-doc --pkg-config=/Users/ec2-user/croot/ffmpeg_1789719343630/_build_env/bin/pkg-config --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/ec2-user/croot/ffmpeg_1789719343630/_build_env/bin/arm64-apple-darwin20.0.0-cla

## Use Whisper to transcribe

In [ ]:
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",   # start small; try whisper-large-v3-turbo later
    device="mps",                   # Apple GPU
    chunk_length_s=30               # process 30s
)

result = asr("../data/audio/reel_001.wav", generate_kwargs={"language": "en"})
print(result["text"])


In [ ]:
type(result["text"])

In [ ]:
for reel in reel_names_list:
    audio_path = f"../data/audio/{reel}.wav"
    transcript_path = f"../data/transcripts/{reel}.txt"

    transcript = asr(audio_path, 
                     generate_kwargs={"language":"en"},
                     return_timestamps=True)

    with open(transcript_path, "w", encoding="utf-8") as file:
        file.write(transcript["text"])
    

### Performance summary
Works really well when speech is present and clear. 

Did not work well in the following situations:
- music only (came up with generic transcripts like:  Thank you for watching, and I'll see you in the next video!) this is a stock phrase since whisper was trained on YouTube videos.
- speech with loud music in the background (skipped the speech with music)
- skipped screams and shouting

### Test long format pipeline

In [ ]:
import torch
import scipy.io.wavfile as wavfile
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"   # swap for "openai/whisper-large-v3-turbo" later
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("mps")

def transcribe(audio_path):
    sr, audio = wavfile.read(audio_path)           # 16 kHz mono int16 from our ffmpeg step
    audio = audio.astype("float32") / 32768.0      # scale to the -1..1 range the model expects

    # truncation=False keeps audio longer than 30 s instead of cutting it off
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt",
                       truncation=False, padding="longest", return_attention_mask=True)
    if inputs.input_features.shape[-1] < 3000:     # clips under 30 s must be padded with silence to exactly 30 s
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt",
                           return_attention_mask=True)
    inputs = {k: v.to("mps") for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(**inputs, return_timestamps=True,
                             language="en", task="transcribe")
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

Path("../data/transcripts").mkdir(parents=True, exist_ok=True)
for reel in reel_names_list:
    transcript_path = Path(f"../data/transcripts/{reel}.txt")
    if transcript_path.exists():
        continue
    transcript_path.write_text(transcribe(f"../data/audio/{reel}.wav"), encoding="utf-8")


### Performance summary
- worked better for reel 10 - could parse speech from music
- no generic youtube texts for reels 12, 13

fails:
- no more text for reel 002 for some reason
- reel 13 was music with vocals but did not recognize music unlike instrumental music in reel 12
- still does not recognize screams in reel 8
- panicked like crazy in reel 4 (sports crowd noises)

### Test on larger whisper model

In [ ]:
model_name = "openai/whisper-large-v3-turbo"   
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("mps")

def transcribe(audio_path):
    sr, audio = wavfile.read(audio_path)           # 16 kHz mono int16 from our ffmpeg step
    audio = audio.astype("float32") / 32768.0      

    # truncation=False keeps audio longer than 30 s instead of cutting it off
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt",
                       truncation=False, padding="longest", return_attention_mask=True)
    if inputs.input_features.shape[-1] < 3000:     # clips under 30 s must be padded with silence to exactly 30 s
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt",
                           return_attention_mask=True)
    inputs = {k: v.to("mps") for k, v in inputs.items()}
    inputs["input_features"] = inputs["input_features"].to(model.dtype)   # match the model (float16)


    with torch.no_grad():
        out = model.generate(**inputs, return_timestamps=True,
                             language="en", task="transcribe")
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

Path("../data/transcripts_large_model").mkdir(parents=True, exist_ok=True)
for reel in reel_names_list:
    transcript_path = Path(f"../data/transcripts_large_model/{reel}.txt")
    if transcript_path.exists():
        continue
    transcript_path.write_text(transcribe(f"../data/audio/{reel}.wav"), encoding="utf-8")


### Performance summary
- managed to add "!"
- Still no transcript for reel 2
- otherwise performed pretty much the same as the smaller model
- takes too long - not worth it (24 min for 14 reels)

### Use Pipeline for reels <30s and "generate" for longer reels

In [5]:
import torch
import scipy.io.wavfile as wavfile
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration, pipeline

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("mps")

# the pipeline reuses the model we just loaded (no second copy in memory)
asr = pipeline("automatic-speech-recognition", model=model,
               tokenizer=processor.tokenizer,
               feature_extractor=processor.feature_extractor, device="mps")

def transcribe(audio_path):
    sr, audio = wavfile.read(audio_path)
    audio = audio.astype("float32") / 32768.0

    if len(audio) / sr <= 30:
        # short clip: plain pipeline, no timestamps
        return asr({"raw": audio, "sampling_rate": sr},
                   generate_kwargs={"language": "en"})["text"].strip()

    # long clip: Whisper's native long-form generation (needs timestamps)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt",
                       truncation=False, padding="longest", return_attention_mask=True)
    inputs = {k: v.to("mps") for k, v in inputs.items()}
    inputs["input_features"] = inputs["input_features"].to(model.dtype)
    with torch.no_grad():
        out = model.generate(**inputs, return_timestamps=True,
                             language="en", task="transcribe")
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

out_dir = Path("../data/transcripts_hybrid")        # change if you want a separate folder
out_dir.mkdir(parents=True, exist_ok=True)
for reel in reel_names_list:
    path = out_dir / f"{reel}.txt"
    if path.exists():
        continue
    path.write_text(transcribe(f"../data/audio/{reel}.wav"), encoding="utf-8")


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


### Performance summary

- has transcript for reel 2
- parsed out speech for reel 10
- best version on the data for now